### Imports

In [35]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.functional as F

# Computer vision library
import torchvision
from torchvision.datasets import MNIST, CIFAR10
from torchvision import transforms

### Loading Data

In [ ]:
# Transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)) # Applying in RGB image
])

In [14]:
trainset = CIFAR10(root = './data', train=True,  download=True, transform=transform)

Files already downloaded and verified


In [15]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [ ]:
trainloader = DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)

### Model

* Pooling: reduces the spatial dimension to reduce the data
  * Average pooling: smoothen the feature
  * Max pooling: conserving prominent features
* Padding: to preserve the information

In [ ]:
# (RGB) => (Batch, Channel, Height, Width)
image = torch.randn(1, 3, 32, 32)

conv = nn.Conv2d(in_channels=3, out_channels=18, kernel_size=3, stride=1, padding=1)
output_conv = conv(image)

print("Input shape:", image.shape)
print("Output shape after Conv2d:", output_conv.shape)

pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
output_pool = pool(output_conv)
print("Output shape after MaxPool2d:", output_pool.shape)

fc1 = nn.Linear(in_features=18 * 16 * 16, out_features=64) # 4608 -> 64
fc2 = nn.Linear(64, 10) # 64 -> 10

# Flattening the output for fully connected layer
output_flat = output_pool.view(-1, 18 * 16 * 16)
print("Output shape after flattening:", output_flat.shape)

output_fc1 = fc1(output_flat)
print("Output shape after fc1:", output_fc1.shape)

output_fc2 = fc2(output_fc1)
print("Output shape after fc1:", output_fc2.shape)

probs = nn.Softmax(dim=1)(output_fc2)
print("Output probabilities after Softmax:", probs)

Input shape: torch.Size([1, 3, 32, 32])
Output shape after Conv2d: torch.Size([1, 18, 32, 32])
Output shape after MaxPool2d: torch.Size([1, 18, 16, 16])
Output shape after flattening: torch.Size([1, 4608])
Output shape after fc1: torch.Size([1, 64])
Output shape after fc1: torch.Size([1, 10])
Output probabilities after Softmax: tensor([[0.0937, 0.0805, 0.1243, 0.0967, 0.0839, 0.1137, 0.1198, 0.1033, 0.0983,
         0.0859]], grad_fn=<SoftmaxBackward0>)


In [34]:
class SimpleCNN(torch.nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        #Input channels = 3, output channels = 18
        self.conv1 = torch.nn.Conv2d(3, 18, kernel_size = 3, stride = 1, padding = 1)
        self.pool = torch.nn.MaxPool2d(kernel_size = 2, stride = 2, padding = 0)
        #4608 input features, 64 output features (see sizing flow below)
        self.fc1 = torch.nn.Linear(18 * 16 * 16, 64)
        #64 input features, 10 output features for our 10 defined classes
        self.fc2 = torch.nn.Linear(64, 10)
    
    def forward(self, x):
        pass